Check if the Twitter dataset from Kaggle is already downloaded, if not, ingest it from Kaggle

In [ ]:
import kagglehub
import os

# Create data directory if it doesn't exist
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

# Define the file path within the data directory
file_path = os.path.join(data_dir, "tesla_2018_to_2020.csv")

# Check if file already exists
if os.path.exists(file_path):
    print(f"File already exists at: {os.path.abspath(file_path)}")
    path = file_path
else:
    # Download latest version if file doesn't exist
    path = kagglehub.dataset_download("hindy51/tesla-tweets")
    # Move the downloaded file to the data directory
    if os.path.exists(path):
        os.rename(path, file_path)
        path = file_path
    print("Downloaded dataset to:", path)

print("Path to dataset:", path)

Display basic information about the dataset

In [ ]:
import pandas as pd
import os

# Load the CSV file into a DataFrame
df = pd.read_csv(file_path)

# Convert created_at to datetime and analyze the time range
df['created_at'] = pd.to_datetime(df['created_at'])

print("Date range in the dataset:")
print("Earliest date:", df['created_at'].min())
print("Latest date:", df['created_at'].max())

In [ ]:
import pandas as pd
import os

# Convert created_at to datetime and analyze the time range
df['created_at'] = pd.to_datetime(df['created_at'])

print("Date range in the dataset:")
print("Earliest date:", df['created_at'].min())
print("Latest date:", df['created_at'].max())

# Get basic statistics about tweets per month
df['month_year'] = df['created_at'].dt.to_period('M')
monthly_counts = df.groupby('month_year').size()

print("\nNumber of tweets per month:")
print(monthly_counts)

# Basic content analysis
print("\nLanguage distribution:")
print(df['lang'].value_counts().head())

print("\nVerified users percentage:")
print(f"{(df['user_verified'].mean() * 100):.2f}%")

print("\nEngagement metrics (mean):")
print("Retweets:", df['retweet_count'].mean())
print("Favorites:", df['favorite_count'].mean())
print("Replies:", df['reply_count'].mean())

Data preprocessing

In [ ]:
# Filter for English tweets and remove unnecessary columns
df_clean = df[df['lang'] == 'en'].copy()

# Basic text preprocessing
import re

def clean_tweet(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^\w\s]', '', text)
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

# Apply cleaning to full_text column
df_clean['cleaned_text'] = df_clean['full_text'].apply(clean_tweet)

# Display sample of original and cleaned tweets
print("Sample of original vs cleaned tweets:")
sample = df_clean[['full_text', 'cleaned_text']].head()
print(sample)

# Get monthly tweet counts after cleaning
df_clean['month_year'] = df_clean['created_at'].dt.to_period('M')
monthly_clean_counts = df_clean.groupby('month_year').size()
print("\nNumber of clean English tweets per month:")
print(monthly_clean_counts)

#Perform sentiment analysis using VADER (Valence Aware Dictionary and sEntiment Reasoner)

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Function to get compound sentiment score
def get_sentiment(text):
    try:
        return analyzer.polarity_scores(text)['compound']
    except:
        return 0

# Apply sentiment analysis
df_clean['sentiment'] = df_clean['cleaned_text'].apply(get_sentiment)

# Calculate monthly average sentiment
monthly_sentiment = df_clean.groupby('month_year')['sentiment'].agg(['mean', 'count'])
monthly_sentiment = monthly_sentiment.sort_index()

print("Monthly sentiment analysis:")
print(monthly_sentiment)

# Basic sentiment statistics
print("\nOverall sentiment statistics:")
print(df_clean['sentiment'].describe())

# Distribution of sentiment
print("\nSentiment distribution:")
print("Positive tweets (> 0):", (df_clean['sentiment'] > 0).mean() * 100, "%")
print("Neutral tweets (= 0):", (df_clean['sentiment'] == 0).mean() * 100, "%")
print("Negative tweets (< 0):", (df_clean['sentiment'] < 0).mean() * 100, "%")

#visualize the trends

In [ ]:
import matplotlib.pyplot as plt

# Create the plot
fig, ax1 = plt.subplots(figsize=(15, 6))

# Plot sentiment line (blue)
ax1.plot(monthly_sentiment.index.astype(str), monthly_sentiment['mean'],
         color='blue', marker='o', linewidth=2, label='Average Sentiment')
ax1.set_xlabel('Month', fontsize=10)
ax1.set_ylabel('Average Sentiment', color='blue', fontsize=10)
ax1.tick_params(axis='x', rotation=45)
ax1.tick_params(axis='y', labelcolor='blue')
ax1.grid(True, alpha=0.3)

# Plot volume (gray bars) on secondary y-axis
ax2 = ax1.twinx()
ax2.bar(monthly_sentiment.index.astype(str), monthly_sentiment['count'],
        alpha=0.3, color='gray', label='Tweet Volume')
ax2.set_ylabel('Tweet Volume', color='gray', fontsize=10)
ax2.tick_params(axis='y', labelcolor='gray')

# Title and legend
plt.title('Tesla Tweet Sentiment and Volume Over Time', fontsize=12, pad=20)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

#Monthly sentiment analysis（positive/negative/neutral）

In [ ]:
# Create sentiment categories
def categorize_sentiment(score):
    if score > 0:
        return 'Positive'
    elif score < 0:
        return 'Negative'
    else:
        return 'Neutral'

# Add sentiment categories to the dataframe
df_clean['sentiment_category'] = df_clean['sentiment'].apply(categorize_sentiment)

# Group by month and sentiment category
monthly_categories = df_clean.groupby(['month_year', 'sentiment_category']).size().unstack(fill_value=0)

# Plot
plt.figure(figsize=(15, 8))
monthly_categories.plot(kind='bar', stacked=True,
                      color=['#1f77b4', '#2ca02c', '#7f7f7f'])  # Blue for negative, green for positive, gray for neutral

plt.title('Distribution of Tesla Tweet Sentiments by Month', fontsize=12, pad=20)
plt.xlabel('Month', fontsize=10)
plt.ylabel('Number of Tweets', fontsize=10)
plt.legend(title='Sentiment')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#Making word clouds for 2020 Jan to Nov

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import numpy as np
from collections import Counter
import re

def preprocess_text(text):
    # Remove URLs, mentions, special characters, and common words
    text = re.sub(r'http\S+|@\S+|#\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    # Remove 'tesla' since it will be in most tweets
    text = text.lower().replace('tesla', '')
    # Remove single characters
    text = ' '.join([word for word in text.split() if len(word) > 1])
    return text

def create_monthly_wordcloud(df, month_year):
    # Filter data for the specific month
    month_data = df[df['month_year'] == month_year]

    # Combine all tweets for the month
    text = ' '.join(month_data['cleaned_text'].apply(preprocess_text))

    # Create and generate a word cloud image
    wordcloud = WordCloud(
        width=800, height=400,
        background_color='white',
        max_words=100,
        max_font_size=100,
        random_state=42
    ).generate(text)

    return wordcloud

# Filter months for 2020 Jan-Nov
months_2020 = [month for month in sorted(df_clean['month_year'].unique())
               if str(month).startswith('2020') and str(month) != '2020-12']

print("Months being processed:", months_2020)  # Debug print to see what months we're getting

# Create subplot grid (4 rows x 3 columns)
plt.figure(figsize=(20, 25))

for idx, month in enumerate(months_2020, 1):
    plt.subplot(4, 3, idx)
    wordcloud = create_monthly_wordcloud(df_clean, month)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'{month}', pad=20)

plt.tight_layout()
plt.show()

#Plot of sales 2018 to 2020 (US sales)
source: https://www.goodcarbadcar.net/tesla-inc-us-sales-figures/

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.dates import MonthLocator, DateFormatter

# Create continuous timeline data
months = pd.date_range(start='2018-01-01', end='2020-12-01', freq='MS')  # MS for month start
sales = [
    # 2018
    6875, 7485, 8820, 6150, 11250, 11062, 16675, 21700, 29975, 20325, 24600, 32600,
    # 2019
    8325, 7650, 14625, 11925, 16350, 25025, 15650, 16025, 23025, 18612, 19301, 18612,
    # 2020
    22350, 20450, 10000, 6624, 14720, 15456, 48846, 43418, 47036, 21591, 17736, 24675
]

# Create the plot
plt.figure(figsize=(15, 7))

# Plot line with markers
plt.plot(months, sales, marker='o', linewidth=2, markersize=6)

# Customize the plot
plt.title('Tesla US Sales (2018-2020)', fontsize=14, pad=20)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Number of Vehicles Sold', fontsize=12)
plt.grid(True, alpha=0.3)

# Format x-axis to show months and years
plt.gcf().autofmt_xdate()  # Rotate and align the tick labels
plt.gca().xaxis.set_major_locator(MonthLocator(interval=3))  # Show every 3 months
plt.gca().xaxis.set_major_formatter(DateFormatter('%Y-%m'))

# Add commas to y-axis labels
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ',')))

# Add horizontal lines for year transitions
plt.axvline(pd.to_datetime('2019-01-01'), color='gray', linestyle='--', alpha=0.5)
plt.axvline(pd.to_datetime('2020-01-01'), color='gray', linestyle='--', alpha=0.5)

# Adjust layout
plt.tight_layout()

plt.show()

## Prediction model
# Input Data

Tesla's monthly US sales numbers from 2018 to 2020.

Twitter data about Tesla from the same period, including: How positive/negative the tweets were (sentiment scores) and how many tweets there were each month (tweet volume).

# What We're Predicting

Tesla's monthly sales numbers for July-December 2020. Using data from January 2018 to June 2020 to make these predictions.

# Features We Created

Sales History: Last month's sales, sales from 2 months ago, sales from 3 months ago, month-to-month sales changes.

Twitter Information: Monthly sentiment scores, monthly tweet counts, 3-month averages of both sentiment and tweet volume.

Time Information: Month of the year (to catch seasonal patterns), year (to catch long-term trends).

# Technology Used

Random Forest: It's like having 100 different decision trees voting on what the sales will be. Each tree looks at different combinations of our features. The model tells us which features are most important for predictions.

# Most important factors:

Month of the year (20.5%)

Previous month's sales (20.4%)

Twitter sentiment (18.3%)

# Results

The model was good at predicting normal sales patterns. But it couldn't predict the huge sales spike in Q3 2020. This makes sense because there wasn't anything similar in the training data.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# First create the timeline data
months = pd.date_range(start='2018-01-01', end='2020-12-01', freq='MS')
sales = [
    # 2018
    6875, 7485, 8820, 6150, 11250, 11062, 16675, 21700, 29975, 20325, 24600, 32600,
    # 2019
    8325, 7650, 14625, 11925, 16350, 25025, 15650, 16025, 23025, 18612, 19301, 18612,
    # 2020
    22350, 20450, 10000, 6624, 14720, 15456, 48846, 43418, 47036, 21591, 17736, 24675
]

# Create base DataFrame with sales data
data = pd.DataFrame({
    'sales': sales
}, index=months)

# Convert PeriodIndex to timestamp if necessary
if isinstance(df_clean['month_year'].dtype, pd.PeriodDtype):
    df_clean['month_year'] = df_clean['month_year'].dt.to_timestamp()

# Add sentiment percentages from your sentiment categories
monthly_sentiments = df_clean.groupby('month_year').agg({
    'sentiment_category': lambda x: (x == 'Positive').mean() * 100,  # % positive
    'sentiment': ['mean', 'count']  # average sentiment and volume
}).round(2)

monthly_sentiments.columns = ['percent_positive', 'mean_sentiment', 'tweet_volume']

# Merge sales data with sentiment data
data = pd.merge(data, monthly_sentiments,
                left_index=True,
                right_index=True,
                how='left')

def create_features(df):
    df = df.copy()

    # Basic features
    df['prev_month_sales'] = df['sales'].shift(1)
    df['sales_change'] = df['sales'] - df['prev_month_sales']

    # Add more lag features
    df['sales_2m_ago'] = df['sales'].shift(2)
    df['sales_3m_ago'] = df['sales'].shift(3)

    # Add sentiment and volume moving averages
    df['sentiment_ma3'] = df['mean_sentiment'].rolling(window=3).mean()
    df['volume_ma3'] = df['tweet_volume'].rolling(window=3).mean()
    df['positive_ma3'] = df['percent_positive'].rolling(window=3).mean()

    # Add seasonal features
    df['month'] = df.index.month
    df['year'] = df.index.year

    return df.dropna()

# Prepare data
data = create_features(data)

# First define the train/test split
train_mask = data.index < pd.Timestamp('2020-06-01')
test_mask = data.index >= pd.Timestamp('2020-06-01')

# Then add the print statements
print("\nTraining Data Examples:")
print("------------------------")
# Let's just show the last 3 months of training data to keep output manageable
for idx in data[train_mask].index[-3:]:
    print(f"\nMonth: {idx.strftime('%Y-%m')}")
    print(f"Sales: {data.loc[idx, 'sales']:,}")
    print(f"Previous Month Sales: {data.loc[idx, 'prev_month_sales']:,}")
    print(f"2 Months Ago Sales: {data.loc[idx, 'sales_2m_ago']:,}")
    print(f"Average Sentiment: {data.loc[idx, 'mean_sentiment']:.2f}")
    print(f"Tweet Volume: {data.loc[idx, 'tweet_volume']:,}")
    print(f"Percent Positive Tweets: {data.loc[idx, 'percent_positive']:.1f}%")
    print("------------------------")

print("\nPrediction Data:")
print("------------------------")
for idx in data[test_mask].index:
    print(f"\nMonth: {idx.strftime('%Y-%m')}")
    print(f"Actual Sales: {data.loc[idx, 'sales']:,}")
    print(f"Previous Month Sales: {data.loc[idx, 'prev_month_sales']:,}")
    print(f"2 Months Ago Sales: {data.loc[idx, 'sales_2m_ago']:,}")
    print(f"Average Sentiment: {data.loc[idx, 'mean_sentiment']:.2f}")
    print(f"Tweet Volume: {data.loc[idx, 'tweet_volume']:,}")
    print(f"Percent Positive Tweets: {data.loc[idx, 'percent_positive']:.1f}%")
    print("------------------------")

# Features for prediction
feature_columns = [
    'mean_sentiment', 'tweet_volume', 'percent_positive',
    'prev_month_sales', 'sales_2m_ago', 'sales_3m_ago',
    'sentiment_ma3', 'volume_ma3', 'positive_ma3',
    'month', 'year'
]

X = data[feature_columns]
y = data['sales']

# Split data: Use data until June 2020 for training
train_mask = data.index < pd.Timestamp('2020-06-01')
test_mask = data.index >= pd.Timestamp('2020-06-01')

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest model
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Print results
print("\nActual vs Predicted Sales for July-November 2020:")
print("\nActual Sales:")
print(data[test_mask]['sales'].values)
print("\nPredicted Sales:")
print(np.array(y_pred))

# Calculate metrics
mse = mean_squared_error(data[test_mask]['sales'].values, y_pred)
r2 = r2_score(data[test_mask]['sales'].values, y_pred)

print(f"\nRoot Mean Squared Error: {np.sqrt(mse):.2f}")
print(f"R-squared Score: {r2:.2f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
})
print("\nFeature Importance:")
print(feature_importance.sort_values('importance', ascending=False))

# Visualize results
plt.figure(figsize=(12, 6))
plt.plot(data[test_mask].index,
         data[test_mask]['sales'].values,
         'b-', label='Actual Sales')
plt.plot(data[test_mask].index,
         y_pred, 'r--', label='Predicted Sales')
plt.title('Tesla Sales: Actual vs Predicted (July-November 2020)')
plt.xlabel('Month')
plt.ylabel('Number of Vehicles Sold')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Optional: Print the actual vs predicted values in a more readable format
comparison = pd.DataFrame({
    'Actual Sales': data[test_mask]['sales'].values,
    'Predicted Sales': y_pred
}, index=data[test_mask].index)
print("\nDetailed Comparison:")
print(comparison)

Applying XGBoost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# First create the timeline data
months = pd.date_range(start='2018-01-01', end='2020-12-01', freq='MS')
sales = [
    # 2018
    6875, 7485, 8820, 6150, 11250, 11062, 16675, 21700, 29975, 20325, 24600, 32600,
    # 2019
    8325, 7650, 14625, 11925, 16350, 25025, 15650, 16025, 23025, 18612, 19301, 18612,
    # 2020
    22350, 20450, 10000, 6624, 14720, 15456, 48846, 43418, 47036, 21591, 17736, 24675
]

# Create base DataFrame with sales data
data = pd.DataFrame({
    'sales': sales
}, index=months)

# Convert PeriodIndex to timestamp if necessary
if isinstance(df_clean['month_year'].dtype, pd.PeriodDtype):
    df_clean['month_year'] = df_clean['month_year'].dt.to_timestamp()

# Add sentiment percentages from your sentiment categories
monthly_sentiments = df_clean.groupby('month_year').agg({
    'sentiment_category': lambda x: (x == 'Positive').mean() * 100,  # % positive
    'sentiment': ['mean', 'count']  # average sentiment and volume
}).round(2)

monthly_sentiments.columns = ['percent_positive', 'mean_sentiment', 'tweet_volume']

# Merge sales data with sentiment data
data = pd.merge(data, monthly_sentiments,
                left_index=True,
                right_index=True,
                how='left')

def create_features(df):
    df = df.copy()

    # Basic features
    df['prev_month_sales'] = df['sales'].shift(1)
    df['sales_change'] = df['sales'] - df['prev_month_sales']

    # Add more lag features
    df['sales_2m_ago'] = df['sales'].shift(2)
    df['sales_3m_ago'] = df['sales'].shift(3)

    # Add sentiment and volume moving averages
    df['sentiment_ma3'] = df['mean_sentiment'].rolling(window=3).mean()
    df['volume_ma3'] = df['tweet_volume'].rolling(window=3).mean()
    df['positive_ma3'] = df['percent_positive'].rolling(window=3).mean()

    # Add seasonal features
    df['month'] = df.index.month
    df['year'] = df.index.year

    return df.dropna()

# Prepare data
data = create_features(data)

# Features for prediction
feature_columns = [
    'mean_sentiment', 'tweet_volume', 'percent_positive',
    'prev_month_sales', 'sales_2m_ago', 'sales_3m_ago',
    'sentiment_ma3', 'volume_ma3', 'positive_ma3',
    'month', 'year'
]

X = data[feature_columns]
y = data['sales']

# Split data: Use data until June 2020 for training
train_mask = data.index < pd.Timestamp('2020-06-01')
test_mask = data.index >= pd.Timestamp('2020-06-01')

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train XGBoost model
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Print results
print("\nActual vs Predicted Sales for July-November 2020:")
print("\nActual Sales:")
print(data[test_mask]['sales'].values)
print("\nPredicted Sales:")
print(np.array(y_pred))

# Calculate metrics
mse = mean_squared_error(data[test_mask]['sales'].values, y_pred)
r2 = r2_score(data[test_mask]['sales'].values, y_pred)

print(f"\nRoot Mean Squared Error: {np.sqrt(mse):.2f}")
print(f"R-squared Score: {r2:.2f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
})
print("\nFeature Importance:")
print(feature_importance.sort_values('importance', ascending=False))

# Visualize results
plt.figure(figsize=(12, 6))
plt.plot(data[test_mask].index,
         data[test_mask]['sales'].values,
         'b-', label='Actual Sales')
plt.plot(data[test_mask].index,
         y_pred, 'r--', label='Predicted Sales')
plt.title('Tesla Sales: Actual vs Predicted (July-November 2020)')
plt.xlabel('Month')
plt.ylabel('Number of Vehicles Sold')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Optional: Print the actual vs predicted values in a more readable format
comparison = pd.DataFrame({
    'Actual Sales': data[test_mask]['sales'].values,
    'Predicted Sales': y_pred
}, index=data[test_mask].index)
print("\nDetailed Comparison:")
print(comparison)

Using LSTM

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# Prepare the data for LSTM
def create_sequences(data, n_steps):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:(i + n_steps)])
        y.append(data[i + n_steps])
    return np.array(X), np.array(y)

# Normalize the data
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data[['sales', 'mean_sentiment', 'tweet_volume', 'percent_positive']])

# Number of time steps to look back
n_steps = 3  # Using 3 months of history to predict the next month

# Create sequences
X, y = create_sequences(data_scaled, n_steps)

# Get the dates for the test period (June 2020 to December 2020)
test_dates = data.index[(data.index >= pd.Timestamp('2020-06-01')) & 
                       (data.index <= pd.Timestamp('2020-12-01'))]

# Get the indices for the test period
test_indices = []
for date in test_dates:
    idx = np.where(data.index == date)[0][0]
    if idx >= n_steps:  # Ensure we have enough history
        test_indices.append(idx - n_steps)

# Get the indices for training (everything before June 2020)
train_indices = np.where(data.index < pd.Timestamp('2020-06-01'))[0]
train_indices = train_indices[train_indices < len(X)]

# Prepare the data
X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

# Reshape data for LSTM [samples, timesteps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], X_train.shape[2]))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], X_test.shape[2]))

# Build LSTM model
model = Sequential([
    LSTM(50, activation='relu', input_shape=(n_steps, X_train.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(50, activation='relu'),
    Dropout(0.2),
    Dense(4)  # Predicting all 4 features
])

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Make predictions
y_pred = model.predict(X_test)

# Inverse transform the predictions and actual values
y_pred_inv = scaler.inverse_transform(y_pred)
y_test_inv = scaler.inverse_transform(y_test)

# Extract sales predictions and actual values
sales_pred = y_pred_inv[:, 0]
sales_actual = y_test_inv[:, 0]

# Calculate metrics
mse = mean_squared_error(sales_actual, sales_pred)
rmse = np.sqrt(mse)
r2 = r2_score(sales_actual, sales_pred)

print(f"\nLSTM Model Results:")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R-squared Score: {r2:.2f}")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot actual vs predicted sales
plt.figure(figsize=(12, 6))
plt.plot(test_dates, sales_actual, label='Actual Sales', color='blue')
plt.plot(test_dates, sales_pred, label='Predicted Sales', color='red', linestyle='--')
plt.title('Tesla Sales: Actual vs Predicted (June 2020 - December 2020)')
plt.xlabel('Date')
plt.ylabel('Number of Vehicles Sold')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Print detailed comparison with dates
comparison = pd.DataFrame({
    'Date': test_dates,
    'Actual Sales': sales_actual,
    'Predicted Sales': sales_pred
})
print("\nDetailed Comparison (June 2020 - December 2020):")
print(comparison)

Ensemble Version of Random Forest, XGBoost and LSTM


Using Stacking here because:
1.	The data likely has complex temporal patterns
2.	Different models capture different aspects of the data:
	- LSTM captures temporal dependencies
	- Random Forest handles non-linear relationships
	- XGBoost is good at feature importance
3.	Stacking can learn how to best combine these different strengths


Stacking works in two levels:
Base Level: Multiple models (LSTM, Random Forest, XGBoost) make predictions independently.
Meta Level: A meta-model (Linear Regression) learns how to best combine these predictions by:
Using the base models' predictions as input features
Learning optimal weights for each model
Finding the best combination to minimize prediction error
The meta-model essentially learns which model performs best in different situations and creates a weighted combination that outperforms individual models. It's like having a "model of models" that learns the strengths of each base model.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# Prepare the data for LSTM
def create_sequences(data, n_steps):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:(i + n_steps)])
        y.append(data[i + n_steps])
    return np.array(X), np.array(y)

# Create lagged features for traditional models
def create_lagged_features(data, n_lags=3):
    df = pd.DataFrame(data)
    for i in range(1, n_lags + 1):
        df[f'lag_{i}'] = df[0].shift(i)
    return df.dropna()

# Normalize the data
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data[['sales', 'mean_sentiment', 'tweet_volume', 'percent_positive']])

# Create separate scaler for sales
sales_scaler = MinMaxScaler()
sales_scaled = sales_scaler.fit_transform(data[['sales']])

# Number of time steps to look back
n_steps = 3

# Get the dates for the test period (June 2020 to December 2020)
test_dates = data.index[(data.index >= pd.Timestamp('2020-06-01')) & 
                       (data.index <= pd.Timestamp('2020-12-01'))]

# Prepare data for LSTM
X_seq, y_seq = create_sequences(data_scaled, n_steps)

# Get indices for test period
test_indices = []
for date in test_dates:
    idx = np.where(data.index == date)[0][0]
    if idx >= n_steps:
        test_indices.append(idx - n_steps)

# Get indices for training
train_indices = np.where(data.index < pd.Timestamp('2020-06-01'))[0]
train_indices = train_indices[train_indices < len(X_seq)]

# Prepare LSTM data
X_train_lstm = X_seq[train_indices]
X_test_lstm = X_seq[test_indices]
y_train_lstm = y_seq[train_indices]
y_test_lstm = y_seq[test_indices]

# Reshape LSTM data
X_train_lstm = X_train_lstm.reshape((X_train_lstm.shape[0], X_train_lstm.shape[1], X_train_lstm.shape[2]))
X_test_lstm = X_test_lstm.reshape((X_test_lstm.shape[0], X_test_lstm.shape[1], X_test_lstm.shape[2]))

# Prepare data for traditional models
data_lagged = create_lagged_features(sales_scaled)
X_trad = data_lagged.drop(0, axis=1)
y_trad = data_lagged[0]

# Convert column names to strings
X_trad.columns = X_trad.columns.astype(str)

# Create a DataFrame with the original dates for the lagged data
lagged_dates = data.index[n_steps:]
data_lagged.index = lagged_dates

# Split for traditional models
train_mask = data_lagged.index < pd.Timestamp('2020-06-01')
test_mask = (data_lagged.index >= pd.Timestamp('2020-06-01')) & (data_lagged.index <= pd.Timestamp('2020-12-01'))

X_train_trad = X_trad[train_mask]
X_test_trad = X_trad[test_mask]
y_train_trad = y_trad[train_mask]
y_test_trad = y_trad[test_mask]

# Build and train LSTM model
lstm_model = Sequential([
    LSTM(50, activation='relu', input_shape=(n_steps, X_train_lstm.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(50, activation='relu'),
    Dropout(0.2),
    Dense(4)
])
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.fit(X_train_lstm, y_train_lstm, epochs=100, batch_size=32, verbose=0)

# Build and train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_trad, y_train_trad)

# Build and train XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train_trad, y_train_trad)

# Make predictions
lstm_pred = lstm_model.predict(X_test_lstm)
rf_pred = rf_model.predict(X_test_trad)
xgb_pred = xgb_model.predict(X_test_trad)

# Inverse transform predictions
lstm_pred_inv = scaler.inverse_transform(lstm_pred)[:, 0]
rf_pred_inv = sales_scaler.inverse_transform(rf_pred.reshape(-1, 1))[:, 0]
xgb_pred_inv = sales_scaler.inverse_transform(xgb_pred.reshape(-1, 1))[:, 0]
y_test_inv = scaler.inverse_transform(y_test_lstm)[:, 0]

# Create meta-features for stacking
meta_features = np.column_stack((lstm_pred_inv, rf_pred_inv, xgb_pred_inv))

# Train meta-model (stacking)
meta_model = LinearRegression()
meta_model.fit(meta_features, y_test_inv)

# Make final predictions using stacking
ensemble_pred = meta_model.predict(meta_features)

# Calculate metrics
def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return rmse, r2

lstm_rmse, lstm_r2 = calculate_metrics(y_test_inv, lstm_pred_inv)
rf_rmse, rf_r2 = calculate_metrics(y_test_inv, rf_pred_inv)
xgb_rmse, xgb_r2 = calculate_metrics(y_test_inv, xgb_pred_inv)
ensemble_rmse, ensemble_r2 = calculate_metrics(y_test_inv, ensemble_pred)

# Print results
print("\nModel Results:")
print(f"LSTM - RMSE: {lstm_rmse:.2f}, R2: {lstm_r2:.2f}")
print(f"Random Forest - RMSE: {rf_rmse:.2f}, R2: {rf_r2:.2f}")
print(f"XGBoost - RMSE: {xgb_rmse:.2f}, R2: {xgb_r2:.2f}")
print(f"Stacking Ensemble - RMSE: {ensemble_rmse:.2f}, R2: {ensemble_r2:.2f}")

# Print stacking weights
print("\nStacking Weights:")
for i, weight in enumerate(meta_model.coef_):
    print(f"Model {i+1} weight: {weight:.4f}")
print(f"Intercept: {meta_model.intercept_:.4f}")

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(test_dates, y_test_inv, label='Actual Sales', color='blue')
plt.plot(test_dates, lstm_pred_inv, label='LSTM', color='red', linestyle='--')
plt.plot(test_dates, rf_pred_inv, label='Random Forest', color='green', linestyle='--')
plt.plot(test_dates, xgb_pred_inv, label='XGBoost', color='purple', linestyle='--')
plt.plot(test_dates, ensemble_pred, label='Stacking Ensemble', color='orange', linewidth=2)
plt.title('Tesla Sales: Actual vs Predicted (June 2020 - December 2020)')
plt.xlabel('Date')
plt.ylabel('Number of Vehicles Sold')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Print detailed comparison
comparison = pd.DataFrame({
    'Date': test_dates,
    'Actual Sales': y_test_inv,
    'LSTM Prediction': lstm_pred_inv,
    'Random Forest Prediction': rf_pred_inv,
    'XGBoost Prediction': xgb_pred_inv,
    'Stacking Ensemble Prediction': ensemble_pred
})
print("\nDetailed Comparison (June 2020 - December 2020):")
print(comparison)